In [2]:
from envinit import Workspace
workspace = Workspace()
workspace.init()

INFO	Workspace initialized successfully.


In [3]:
from vllm import LLM, SamplingParams
# import vllm.envs as envs
# envs.VLLM_HOST_IP = "0.0.0.0"
import logging
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from PIL import Image
import pandas as pd
import requests
import torch
import re
import json
from tqdm import tqdm

logging.basicConfig(level=logging.ERROR)
logging.getLogger("vllm").setLevel(logging.ERROR)

INFO 05-18 11:38:51 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
model_id = "google/gemma-3-27b-it"
llm = LLM(model=model_id,
    max_model_len=10000,
    dtype=torch.bfloat16,
    swap_space=32,
    enable_prefix_caching=True
)

processor = AutoProcessor.from_pretrained(model_id, use_fast=False)

In [7]:
df = pd.read_csv("zaytung_full_content_retry.csv")

In [8]:
# remove comments
df["Content"] = df["Content"].str.split("Vahit").str[0].str.strip()

In [6]:
def get_article(id, df):
    title = df.iloc[id, 1]
    body = df.iloc[id, 2]

    return title, body

In [7]:
def extract_json(response):

    blocks = re.findall(r"```json\n(.*?)\n```", response, re.DOTALL)

    json_objects = []
    for block in blocks:
        try:
            parsed = json.loads(block.strip())
            json_objects.append(parsed)
        except json.JSONDecodeError as e:
            print("Failed to parse JSON block.")

    return json_objects, blocks

In [8]:
from string import Template
prompt = Template("""Aşağıdaki haber bir yapay zeka (LLM) outputu olduğunu düşün. Reverse engineering projesi için yapay zeka'dan bu haberi alabilmek için input promptu oluşturman lazım. Prompt türkçe olmalı.

Bunun dışında, haberden metadata çıkarman gerekiyor.

1- Ton/Stil keywordler (3-5 keyword).
2- Haber içinde kullanılan varlıklar (Nesneler, Kişiler, Kurumlar).
3- Tema belirleyici keywordlar (3-5 keyword)

Son output JSON formatında olması lazım, aşağıdaki template kullan:
{   "Prompt": "Write prompt here",
    "Style": ["style keyword 1", "style keyword 2", "style keyword 3", ...]
    "PER": ["person 1", "person 2", "person 3", ...],
    "ORG": ["organization 1", "organization 2", "organization 3", ...],
    "OBJ": ["object keyword 1", "object keyword 2", "object keyword 3", ...] // dont add counts
    "MISC": ["misc keyword 1", "misc keyword 2", "misc keyword 3", ...]
}

Son outputu vermeden önce düşün ve düşünme kısmı ile başlayın.

Başlık: $title

Haber metini: $body""")

In [9]:
template = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt.substitute(title="baasss", body="haberrrr") },

        ]
    },
]

inputs = processor.apply_chat_template(
	template, add_generation_prompt=True
)

In [ ]:
sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=1024)

tagged_df = pd.DataFrame(columns=["Prompt", "Style", "PER", "ORG", "OBJ", "MISC"])

for id in tqdm(range(df.shape[0])):
    
    title, body = get_article(id, df)

    template = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt.substitute(title=title, body=body) },

            ]
        }
    ]

    inputs = processor.apply_chat_template(
        template, add_generation_prompt=True
    )

    with torch.inference_mode(): 
        generations = llm.generate(inputs, sampling_params)

    json_output = extract_json(generations[0].outputs[0].text)[0][0]
    tagged_df.loc[len(tagged_df)] = json_output

In [ ]:
templates = []
for id in tqdm(range(df.shape[0])):
    
    title, body = get_article(id, df)

    template = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt.substitute(title=title, body=body) },
            ]
        }
    ]

    templates.append(template)

inputs = processor.apply_chat_template(
        templates, add_generation_prompt=True
    )

sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=1024)
with torch.inference_mode(): 
        generations = llm.generate(inputs, sampling_params)

tagged_df = pd.DataFrame(columns=["Prompt", "Style", "PER", "ORG", "OBJ", "MISC"])
jsons = []
for generation in generations:
    json_output = extract_json(generation.outputs[0].text)[0][0]
    jsons.append(json_output)

for i in range(len(jsons)):
    tagged_df.loc[len(tagged_df)] = jsons[i]

tagged_df.to_excel("annotated.xlsx", index=False)